# Collapse creation review: cuts and target islands

Review the **131 unresolved unit/compound cut cases**, prioritizing the **35 with distances over 100 m**, and compare island interventions with independent-unit target sets. Small coordinate offsets are not automatically missing cuts.

This notebook reads saved graphs and audit tables. It does not regenerate networks, delete duplicate files, change calibration, or exclude results. Human decisions are saved separately only when you explicitly enable the save cell.

**Identity rule:** a state name such as `13 / S039_unit_33` means network 13, child state S039, and **unit 33 in S000_base**. Newly detected child unit IDs must not be used to identify the original intervention. An island entry can point to an independent-state folder because it is already an alias.

**Review order:** (1) inspect required boundaries and stage geometry; (2) inspect target island sets; (3) retain the independent case as canonical for duplicate interventions and track anchoring repairs separately; (4) review creation of distinct interventions. Same target alone does not prove that either resulting graph is correct.

**Recorded user decisions (8 October 2026):**
- Retain independent cases as canonical when an island intervention duplicates their target; anchoring repairs are a separate task.
- Retain the approved island_41_099 absorption of the tiny island_41_100 artifact.
- **41 / S016_unit_4: missing anchors 5 and 7**, confirmed by the user; repair required.

These decisions update review records, not production graph files.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import plotly.io as pio
from IPython.display import display

REPO = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / 'RAPID/src/rapid_tools').exists())
sys.path.insert(0, str(REPO / 'RAPID/src'))
from rapid_tools.collapse_creation_review import (
    AUDIT, OUT, load_review_data, cut_queue, case_metadata, boundary_table,
    plot_case, stage_paths, case_row, save_review, saved_reviews, DECISIONS,
)
pio.renderers.default = 'plotly_mimetype'
registry, boundaries, unit_classes = load_review_data()
print('Review data:', OUT)
print('All distances are metres in the source projected CRS.')
display(unit_classes.groupby('review_class').size().rename('unique_unit_or_compound_outputs').to_frame())

## 1. Cut-location queues

- **priority:** 35 distinct unit/compound outputs with a required boundary over 100 m from any child node.
- **other_candidates:** 96 outputs whose nearest projected boundary location is not a node, with nearest nodes within 100 m.
- **displaced_controls:** 824 outputs with nodes at their nearest projected boundary locations. A break exists there, but its parent identity/connectivity still needs checking.
- **exact_controls:** 948 outputs retaining all original boundary coordinates.

These classes describe final geometry, not an automatic verdict about the realized-to-anchored transformation. Different aliases of one physical output are grouped; the full `states` column retains their identities.

In [ ]:
QUEUE_KIND = 'priority'  # priority / other_candidates / all_candidates / displaced_controls / exact_controls
NETWORK = None           # e.g. '41', or None for all networks
queue = cut_queue(QUEUE_KIND, NETWORK)
print(f'{len(queue)} unique outputs in this selection')
display(queue[['example', 'states', 'review_class', 'fingerprint']])

## 2. Inspect one state and its original boundaries

Start with 41/S016_unit_4. Then try 13/S039_unit_33, 15/S135_unit_144 and the displaced-cut control **05/S030_unit_37** (the output reused by island_05_038).

The panels show **parent**, **realized assembly before anchoring**, and **final anchored**. Where only raw RivGraph is retained, its panel is explicitly labelled a proxy. Where historical retention removed pre-anchor data, the panel says unavailable; no final graph is substituted for missing evidence.

Purple crosses are **parent boundaries of the recorded domain**. For a wrong-domain island these are not the correct expected boundaries; green diamonds can show separately specified expected boundaries. Dots are actual graph nodes; hover shows their IDs. Orange shading is the recorded replacement domain, and green shading marks target islands when identified. For a wrong island domain, orange and green may not coincide.

Check: Does each required boundary still represent a graph break? Does it connect the appropriate interior and exterior reaches? Does cleanup remove a previously valid replacement? Do exterior connectors remain separate?

In [ ]:
EXAMPLE = '41'
STATE = 'S016_unit_4'
display(case_metadata(EXAMPLE, STATE))
display(boundary_table(EXAMPLE, STATE))
display(pd.DataFrame(stage_paths(case_row(EXAMPLE, STATE)), columns=['stage', 'links_path', 'nodes_path']))
fig = plot_case(EXAMPLE, STATE)
fig.show()

### Boundary close-up

Choose an original `parent_node_id` from the table above. A 7 m offset is visible at this scale. Determine whether the nearby dot is the intended cut, rather than treating any nearby junction as a valid match.

In [ ]:
ANCHOR = 5
RADIUS_M = 80
plot_case(EXAMPLE, STATE, anchor=ANCHOR, radius_m=RADIUS_M).show()

### Optional dropdown browser

Enable this cell for a dropdown browser of either cut queue. The editable cells above work without widgets. The browser does not save decisions or change the `EXAMPLE`/`STATE` variables used by the save cell.

`ipywidgets` is included in the river-hierarchy-rivgraph environment specification. After an environment update, restart the notebook kernel and run the setup cells again.

In [ ]:
ENABLE_BROWSER = True
if ENABLE_BROWSER:
    try:
        import ipywidgets as widgets
    except ModuleNotFoundError as exc:
        raise RuntimeError("This kernel needs ipywidgets. Run %pip install ipywidgets, restart the kernel, then run the setup cells again.") from exc
    mode = widgets.Dropdown(options=['priority', 'other_candidates', 'all_candidates', 'displaced_controls', 'exact_controls'], value=QUEUE_KIND, description='Queue')
    selector = widgets.Dropdown(description='State', layout=widgets.Layout(width='95%'))
    output = widgets.Output()
    def populate(*_):
        q = cut_queue(mode.value, NETWORK)
        selector.options = [(f"{r.example} / {r.states}", (r.example, r.representative_state)) for r in q.itertuples()]
    def show_selected(*_):
        if selector.value is None:
            return
        ex, state = selector.value
        with output:
            output.clear_output(wait=True)
            display(case_metadata(ex, state))
            display(boundary_table(ex, state))
            display(plot_case(ex, state))
    mode.observe(populate, names='value')
    selector.observe(show_selected, names='value')
    populate()
    display(widgets.VBox([mode, selector, output]))

## 3. Target-island census and duplicate flags

For island states, the declared target is the island ID in the state name. For independent/compound states, a preliminary intended set is inferred from islands contained by the selected parent faces; **actual filled targets are checked separately**.

The census tests a known original dry pixel in every frozen island against each output mask. Every fully filled island must pass that witness test; full island-pixel coverage is then verified. Where masks were removed by retention policy, only preserved **exact `geometry_role=edit, action=add` pixel polygons** may reconstruct the edit. Buffered footprints are never accepted as actual edits. Missing evidence is shown as an error, not silently treated as no targets.

Partial island edits are checked on witnessed islands and around the selected parent section (and all retained exact edit polygons when those are used). Partial edits elsewhere and full-network bank-mask equality are not universally certified. Thus target equivalence flags duplicated interventions; it does not by itself authorize deleting files or choosing a canonical graph.

An island result filling multiple islands, the wrong island, or part of another island must be reviewed. The pre-existing manual 41_099/41_100 composite intentionally absorbs a 16-pixel artifact; the user has confirmed that this absorption is correct and must be retained. An independent state with multiple targets cannot be treated as equivalent to one island collapse.

In [ ]:
targets = pd.read_csv(OUT / 'target_inventory.csv', dtype={'example': str})
island_queue = pd.read_csv(OUT / 'island_review_queue.csv', dtype={'example': str})
pairs = pd.read_csv(OUT / 'island_independent_target_pairs.csv', dtype={'example': str})
print('Target-audit entries:', len(targets), 'Evidence errors:', targets.target_error.notna().sum())
display(targets.groupby(['family', 'actual_target_count'], dropna=False).size().rename('entries').to_frame())
display(island_queue.groupby('status').size().rename('island_entries').to_frame())
# These rows remain unresolved; a missing historical mask is not a correctness verdict.
display(targets[targets.target_error.notna()][['example', 'family', 'state', 'target_error']])

### Duplicate interventions and which output to retain

`already_reused=True` means the island references the same saved source as that unit. `same_output` only compares hydraulic fingerprints; separately generated graphs can have equal fingerprints. `island_separately_generated=True` identifies a separate island run (rather than an alias). A singleton match for such a run flags a duplicated intervention for the same singleton target set and parent context. **Approved workflow: retain the independent case as canonical and flag the separate island case as redundant. Repair anchoring independently of this identity decision.** In particular, retain 41/unit 4 as canonical, with its missing anchors explicitly recorded as a repair task. This is not a claim that its current graph is correct.

The same-target pairs include any partial-edit counts, so you can distinguish clean singleton edits from a matching full island accompanied by additional partial changes.

In [ ]:
separate_pairs = pairs[pairs.island_separately_generated].copy()
print('Separately generated island entries with a matching independent singleton target:',
      len(separate_pairs[['example', 'island_state']].drop_duplicates()))
display(separate_pairs[['example', 'island_state', 'independent_state', 'target_island',
                        'island_partial_count', 'independent_partial_count',
                        'island_domain_wrong', 'unit_large_anchor_flag', 'action']])

## 4. Island creation review

Start with island_41_005 to compare the realized replacement with the final cleanup. Then inspect island_41_006 and its same-target independent state S016_unit_4. The whole-compound S076_unit_3 targets both islands and is not an equivalent single-island intervention.

For every distinct island intervention: verify exactly one intended island was filled, the selected processing domain contains it, the realized replacement survives cleanup, and the required interfaces/exterior reaches are preserved. A valid DAG or expected loop reduction alone does not establish correctness.

In [ ]:
ISLAND_EXAMPLE = '41'
ISLAND_STATE = 'island_41_005'
display(island_queue[(island_queue.example == ISLAND_EXAMPLE) & (island_queue.state == ISLAND_STATE)])
display(targets[(targets.example == ISLAND_EXAMPLE) & (targets.state == ISLAND_STATE)][
    ['state', 'declared_target_islands', 'actual_full_target_islands', 'partial_target_islands',
     'target_set_agrees', 'target_error']])
display(pairs[(pairs.example == ISLAND_EXAMPLE) & (pairs.island_state == ISLAND_STATE)])
display(case_metadata(ISLAND_EXAMPLE, ISLAND_STATE))
# Known 41/005 specification: the large island requires the enclosing 4 → 6 domain.
EXPECTED_ISLAND_ANCHORS = [4, 6] if (ISLAND_EXAMPLE, ISLAND_STATE) == ('41', 'island_41_005') else None
plot_case(ISLAND_EXAMPLE, ISLAND_STATE, expected_anchors=EXPECTED_ISLAND_ANCHORS).show()

In [ ]:
# Same intended target, different existing outputs: inspect both.
COMPARE_EXAMPLE = '41'
COMPARE_ISLAND = 'island_41_006'
COMPARE_UNIT_STATE = 'S016_unit_4'
display(pairs[(pairs.example == COMPARE_EXAMPLE) & (pairs.island_state == COMPARE_ISLAND)])
plot_case(COMPARE_EXAMPLE, COMPARE_ISLAND).show()
# Change STATE in section 2 to compare another matching unit, including its boundary close-ups.

## 5. Record a decision

Set the exact case and reviewer below, then explicitly enable saving. This appends a review record under this investigation only; it never deletes graphs or modifies production exclusion lists. Record evidence and the chosen canonical state when recommending reuse. Canonical independent selection does not certify current anchoring. Extra bank/mask changes remain a separate creation check.

Allowed cut decisions: correct_projected_cut, missing_cut, wrong_connection, needs_investigation.

Allowed island decisions: retain_independent_canonical, approved_manual_composite_retain, reuse_validated_unit_output, incorrect_target_set, incorrect_creation, valid_distinct_intervention, needs_investigation.

In [ ]:
SAVE_REVIEW = False
REVIEW_EXAMPLE = EXAMPLE
REVIEW_STATE = STATE
REVIEW_KIND = 'cut'
DECISION = 'needs_investigation'
REVIEWER = ''
NOTES = ''
CANONICAL_STATE = None
if SAVE_REVIEW:
    display(save_review(REVIEW_EXAMPLE, REVIEW_STATE, REVIEW_KIND, DECISION,
                        NOTES, REVIEWER, canonical_state=CANONICAL_STATE))
display(saved_reviews())

## 6. Reproduce the census / exported files

The supplied census is a saved snapshot. To refresh it after geometry changes, use a new output snapshot or deliberately invalidate the investigation's cached network tables; cached successful tables are not a live production monitor.

```bash
/opt/anaconda3/envs/river-hierarchy-rivgraph/bin/python RAPID/investigations/review_collapse_targets.py
```

Files: `target_inventory.csv`, `target_pixel_coverage.csv`, `island_independent_target_pairs.csv`, `island_review_queue.csv`, and optional `human_reviews.jsonl` under `RAPID/outputs/collapse_creation_review_20261008`.

Repair acceptance: target identity → prescribed parent interfaces → verified replacement cuts → final exterior preservation → canonical independent reuse with anchoring repairs tracked separately → regenerated hydraulic inputs and reruns where needed. This notebook provides the evidence and review records; it does not implement those repairs.